# Stage 3, step 2: token-level embeddings for the BioLangFusion encoders

Stage 1 stored one mean-pooled vector per sequence. The BioLangFusion heads read the whole
token sequence, so this step runs the three frozen encoders over the rows
[step 1](01_blf_datasets.ipynb) selected and keeps every token's final hidden state.

**This notebook is a job, not an analysis.** It loads models and writes tens of gigabytes,
so it is saved without outputs and is meant to be run one benchmark at a time. Nothing here
interprets a result; the first result comes in step 3, when the reproduced heads are scored.

The cache it writes is resumable. Work is done in shards of a few hundred sequences, each
written atomically, and a rerun skips the shards already on disk. If the machine sleeps,
the kernel dies, or you stop it on purpose, run the cell again and it continues.


---

## Setup

### Imports

As in Stage 1, `PYTORCH_ENABLE_MPS_FALLBACK` is set before PyTorch is imported so that any
operation the Apple GPU backend lacks runs on the CPU. The path block finds
`Code/shared_code/` from this notebook's folder.


In [ ]:
# Standard library
import os
from pathlib import Path

# Run any operation the Apple GPU backend lacks on the CPU; must precede importing torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Data handling and pretrained encoders
import numpy as np
import pandas as pd
import torch

# Shared project code in Code/shared_code; this notebook lives in Code/stage3
import sys
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import datasets, encoders
from shared_code.fusion import token_embeddings as tokens

### Settings

`BENCHMARK` selects one of the four registered benchmarks; change it and rerun the notebook
for each. The three encoders are BioLangFusion's own, frozen and unchanged: Phase B swaps
come later and write their own caches.

`CHUNK_SIZE` sets how many sequences go in one shard. Smaller shards mean less lost work if
a run is interrupted and a smaller peak in memory; larger shards mean fewer files. A few
hundred suits a laptop.

The cache lives in `Code/stage3/embeddings/<benchmark>/<encoder>/`, which `.gitignore`
already excludes, so nothing here is ever committed.


In [ ]:
BENCHMARK = "cov_vac_degradation"   # one of datasets.BLF_BENCHMARKS; change and rerun
ENCODER_KEYS = ["ntv2_100m", "rnafm", "esm2_8m"]   # BioLangFusion's own three, frozen
CHUNK_SIZE = 256        # sequences per shard
DTYPE = np.float16      # halves the cache; frozen features do not need float32
CACHE_ROOT = Path("embeddings")   # Code/stage3/embeddings/, ignored by Git

DEVICE = encoders.select_device()
ROWS = datasets.blf_benchmark_rows(BENCHMARK)
ENCODERS = [encoders.ENCODERS[k] for k in ENCODER_KEYS]
print(f"{datasets.DATASETS[BENCHMARK].label}: {len(ROWS)} sequences, device {DEVICE}")
print(f"cache: {(CACHE_ROOT / BENCHMARK).resolve()}")

### What this will cost

The token count follows from the sequence lengths and each encoder's tokenization: one token
per nucleotide for RNA-FM, one per codon for ESM-2, one per six nucleotides for Nucleotide
Transformer v2. So the size of the cache is known before any model loads.

Run the cell below to see it for your selection. Across all four benchmarks the total is
about 34 GB in float16, and mRNA stability is 26 GB of that. The other three come to 8 GB
together, which is why they are the ones to do first.


In [ ]:
# Token counts, and so cache size, follow from the sequence lengths alone.
def cost_table(keys):
    rows = []
    for key in keys:
        seqs = datasets.blf_benchmark_rows(key)["sequence"].tolist()
        row = {"benchmark": datasets.DATASETS[key].label, "sequences": len(seqs)}
        total = 0
        for encoder in ENCODERS:
            n = sum(tokens.expected_tokens(encoder, s) for s in seqs)
            width = {"ntv2_100m": 512, "rnafm": 640, "esm2_8m": 320}[encoder.key]  # confirmed at load
            size = n * width * np.dtype(DTYPE).itemsize / 1e9
            row[f"{encoder.key} GB"] = round(size, 2)
            total += size
        rows.append({**row, "total GB": round(total, 2)})
    return pd.DataFrame(rows).set_index("benchmark")

COST = cost_table(datasets.BLF_BENCHMARKS)
print(f"selected: {COST.loc[datasets.DATASETS[BENCHMARK].label, 'total GB']} GB"
      f"   all four: {COST['total GB'].sum():.1f} GB")
COST

---

## A smoke test before the long run

Three things can be wrong in a way that only shows up after hours of work: an encoder could
truncate an input, the stored token count could disagree with the sequence, or the protein
stream could be the wrong length for the codon grid. The next cell extracts a handful of
sequences and checks all three, in under a minute.

The last of those checks deserves a note. Stage 1 translated a coding sequence up to its
first stop codon, which is right for a pooled protein vector. It is wrong here: the codon
grid needs exactly one protein token per codon, and that rule gives one fewer for a clean
coding sequence and far fewer for a fragment with an internal stop. On CoV-Vac, whose
81-nucleotide pieces are cut from a construct rather than being coding sequences, it gives a
full reading frame for only 0.7% of rows and nothing at all for 6.2%.

So the protein stream here uses `grid_protein`, which translates every complete codon and
writes stop codons as `X`. That is deviation 4 from step 1, now settled. BioLangFusion does
not say how it fed such fragments to ESM-2, so this choice is ours and is recorded.


In [ ]:
# Eight sequences through every encoder, checking what a long run cannot afford to get wrong.
sample = ROWS["sequence"].tolist()[:8]
checks = []
for encoder in ENCODERS:
    texts = tokens.inputs_for_rows(encoder, sample)
    tokenizer, model = encoders.load_encoder(encoder, DEVICE)
    try:
        model.eval()
        states = [tokens.token_states(t, tokenizer, model, encoder.max_len, DTYPE) for t in texts]
    finally:
        del model, tokenizer
    want = [tokens.expected_tokens(encoder, s) for s in sample]
    got = [s.shape[0] for s in states]
    grid = [len(t) == len(s) // 3 for t, s in zip(texts, sample)] if encoder.modality == "protein" else [True] * len(sample)
    checks.append({
        "encoder": encoder.label,
        "width": states[0].shape[1],
        "dtype": str(states[0].dtype),
        "tokens (first 3)": str(got[:3]),
        "token counts match": "ok" if got == want else f"MISMATCH want {want[:3]} got {got[:3]}",
        "protein is one token per codon": "ok" if all(grid) else "MISMATCH",
    })
pd.DataFrame(checks).set_index("encoder")

Every row should report `ok`. A truncation error means the rows being
embedded are not the ones step 1 selected. A token-count mismatch means an encoder tokenizes
differently from the rate `expected_token_count` assumes, which would need checking before
the grid can rely on it.


---

## Extract

This is the long cell. It handles one encoder at a time and prints a line per shard, so
progress is visible and an interruption costs at most one shard.

Expect roughly: CoV-Vac in minutes, fungal expression and *E. coli* in a few hours each,
mRNA stability overnight. Leave the machine plugged in and stop it sleeping; on a Mac,
`caffeinate -i` in a terminal keeps it awake while a process runs.


In [ ]:
# The long cell. Resumable: rerun after an interruption and it continues.
for encoder in ENCODERS:
    cache_dir = CACHE_ROOT / BENCHMARK / encoder.key
    texts = tokens.inputs_for_rows(encoder, ROWS["sequence"].tolist())
    print(f"{encoder.label} -> {cache_dir}")
    cache, computed = tokens.extract(encoder, texts, cache_dir, DEVICE,
                                     chunk_size=CHUNK_SIZE, dtype=DTYPE)
    print(f"  {'extracted' if computed else 'already complete'}: {len(cache)} sequences, "
          f"{cache.meta['total_tokens']:,} tokens, width {cache.width}, "
          f"{cache.meta['bytes'] / 1e9:.2f} GB\n")

## Verify the finished cache

The check that matters is per row, not in aggregate: every sequence must have exactly the
tokens its length implies for that encoder. An aggregate byte count would hide a single
sequence that was silently truncated or dropped.


In [ ]:
# Per-row verification: every sequence must have exactly the tokens its length implies.
report = []
for encoder in ENCODERS:
    cache = tokens.summarize(CACHE_ROOT / BENCHMARK / encoder.key)
    want = np.array([tokens.expected_tokens(encoder, s) for s in ROWS["sequence"]])
    same_rows = len(cache) == len(ROWS)
    matches = bool(same_rows and np.array_equal(np.asarray(cache.lengths), want))
    report.append({
        "encoder": encoder.label,
        "rows": f"{len(cache)} / {len(ROWS)}",
        "all rows match": matches,
        "tokens": f"{cache.meta['total_tokens']:,}",
        "width": cache.width,
        "GB on disk": round(sum(p.stat().st_size for p in (CACHE_ROOT / BENCHMARK / encoder.key).glob("shard_*.npz")) / 1e9, 2),
        "first row shape": str(cache[0].shape),
    })
pd.DataFrame(report).set_index("encoder")

With `all rows match` true for all three encoders, this benchmark is
ready for step 3. Record the run in [docs/run-records.md](../../docs/run-records.md) with the
date, device, and the printed sizes, as Stage 1 runs were recorded.


---

## Order of work, and what step 3 needs

Do the benchmarks in this order, smallest first, so a mistake costs minutes rather than a
night:

1. `cov_vac_degradation`, about 0.3 GB. Proves the pipeline end to end.
2. `fungal_expression`, about 3.4 GB. This is the benchmark step 3 checks the build against,
   because its published numbers are clean: 0.804 for the best single encoder, 0.824 fused,
   and no sequence appears in more than one split.
3. `ecoli_proteins`, about 4.3 GB. The one classification benchmark.
4. `mrna_stability`, about 26 GB. Leave it running overnight.

Step 3 then reads these caches and builds the BioLangFusion pipeline on top: the codon-grid
resampling, the three fusion heads, and the fixed TextCNN head. It needs from here, per
benchmark and per encoder, a token cache whose rows line up by position with
`blf_benchmark_rows(benchmark)`, which is what the verification above establishes.

One thing step 3 must decide and this step deliberately does not: how a stream whose tokens
do not divide evenly into the grid is resampled. Nucleotide Transformer v2 gives one token
per six nucleotides, which is two codons, so a sequence whose codon count is odd leaves a
half token at the end. BioLangFusion's transposed convolution assumes an exact factor of two.
That is a modelling choice, so it belongs with the model.
